In [1]:
import subprocess, sys, shutil, json, hashlib
from pathlib import Path
subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics'])
import torch, yaml, pandas as pd
from ultralytics import YOLO
assert torch.cuda.is_available(), 'فعّلي GPU'

INPUT = Path('/kaggle/input'); OUT = Path('/kaggle/working/out'); OUT.mkdir(parents=True, exist_ok=True)
CLASSES = ['D00','D10','D20','D40']
bad = [p.name for p in INPUT.glob('*/*') if 'local' in p.name.lower() or 'my-photos' in p.name.lower()]
assert not bad, f'احذفي المدخلات المحلية: {bad}'

SRC = next(p for p in INPUT.rglob('sample9000') if (p/'train/images').is_dir() and (p/'val/images').is_dir())
ROOT = Path('/tmp/data')
for s in ['train','val']: shutil.copytree(SRC/s, ROOT/s, dirs_exist_ok=True)
DATA = ROOT/'data.yaml'
DATA.write_text(yaml.safe_dump(dict(path=str(ROOT), train='train/images', val='val/images', names=CLASSES)))
print('SRC:', SRC)
print('train', len(list((ROOT/'train/images').glob('*.jpg'))), 'val', len(list((ROOT/'val/images').glob('*.jpg'))))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
SRC: /kaggle/input/notebooks/sarvii7/roadscan-03-sample9000-v2/sample9000
train 8999 val 1977


In [2]:
MODEL = 'yolo11m.pt'     # لو الوقت أقل من 6 ساعات: 'yolo11s.pt'
TRAIN_HOURS = 6.0        # عدّليه حسب وقتك
net = YOLO(MODEL)
net.train(
    data=str(DATA), imgsz=960, epochs=60, time=TRAIN_HOURS, patience=20,
    batch=8, optimizer='AdamW', lr0=0.001, lrf=0.01, cos_lr=True,
    weight_decay=0.0005, warmup_epochs=3,
    mosaic=1.0, close_mosaic=10, mixup=0.1, scale=0.7, translate=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, fliplr=0.5, flipud=0.0,
    seed=42, device=0, workers=2, amp=True,
    project='/kaggle/working/training', name='run24_960', exist_ok=True)
NEW = Path('/kaggle/working/training/run24_960/weights/best.pt')
assert NEW.is_file(), 'ما انحفظ best.pt'
del net; torch.cuda.empty_cache()

Ultralytics 8.4.195 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/tmp/data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=run24_960, nbs=64, nms=None, opset=None, optimize=False, optimizer=Adam

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/38      9.06G      2.332      3.223       2.32         34        960: 100% ━━━━━━━━━━━━ 1125/1125 2.2it/s 8:26
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 124/124 2.4it/s 51.4s
                   all       1977       4650     0.0495     0.0815     0.0257    0.00703

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/39      9.06G       2.27      3.083      2.241         32        960: 100% ━━━━━━━━━━━━ 1125/1125 2.2it/s 8:26
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 124/124 2.4it/s 51.1s
                   all       1977       4650     0.0736      0.106     0.0248    0.00568

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/39      9.06G      2.211      2.968      2.192         23        960: 100% ━━━━━━━━━━━━ 1125/1125 2.2it/s 8:26
                 Class     Images  Instances

In [3]:
cands = {'new_run24': NEW}
for p in INPUT.rglob('*.pt'):
    if p.name == 'run17_best_finetuned.pt': cands['17'] = p
    if p.name == 'best.pt' and '14-roadscan-team-v2' in str(p): cands['14'] = p
print(cands)

rows = []
for name, w in cands.items():
    net = YOLO(str(w))
    for sz in [640, 960]:
        m = net.val(data=str(DATA), split='val', imgsz=sz, batch=8, conf=0.001, iou=0.7,
                    max_det=300, plots=False, verbose=False)
        rows.append(dict(model=name, imgsz=sz, mAP50=m.box.map50, mAP50_95=m.box.map, P=m.box.mp, R=m.box.mr))
        print(rows[-1], flush=True)
    del net; torch.cuda.empty_cache()
final = pd.DataFrame(rows); final.to_csv(OUT/'final_comparison.csv', index=False); display(final)

best = final.sort_values('mAP50_95', ascending=False).iloc[0]
winner = cands[best.model]
shutil.copy2(winner, OUT/'final_weights.pt')
cfg = dict(model=best.model, imgsz=int(best.imgsz), conf=0.46, nms_iou=0.7, max_det=300, matching_iou=0.5,
           sha256=hashlib.sha256((OUT/'final_weights.pt').read_bytes()).hexdigest(),
           val_mAP50=float(best.mAP50), val_mAP50_95=float(best.mAP50_95),
           selection='China validation only (mAP50-95)', local_images_used=False)
(OUT/'final_config.json').write_text(json.dumps(cfg, indent=2))
print('الفائز:', cfg)

{'new_run24': PosixPath('/kaggle/working/training/run24_960/weights/best.pt'), '17': PosixPath('/kaggle/input/notebooks/rimasabunasser/17-roadscan-gentle-finetuning/run17_model_comparison/run17_best_finetuned.pt'), '14': PosixPath('/kaggle/input/notebooks/rimasabunasser/14-roadscan-team-v2-training/runs_team_v2/yolo11s_640_seed42/weights/best.pt')}
Ultralytics 8.4.195 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11m summary (fused): 125 layers, 20,033,116 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1867.8±488.1 MB/s, size: 79.0 KB)
val: Scanning /tmp/data/val/labels.cache... 1977 images, 43 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1977/1977 691.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 248/248 3.9it/s 1:03
                   all       1977       4650      0.334      0.339      0.262      0.101
Speed: 0.9ms preprocess, 28.3ms inference, 0.0ms l

,model,imgsz,mAP50,mAP50_95,P,R
0,new_run24,640,0.261985,0.101472,0.334468,0.338936
1,new_run24,960,0.225216,0.079107,0.316066,0.326864
2,17,640,0.959300,0.657899,0.917367,0.925576
3,17,960,0.869372,0.469599,0.810501,0.840997
4,14,640,0.227095,0.087325,0.297812,0.304142
5,14,960,0.140900,0.040336,0.230590,0.232276


الفائز: {'model': '17', 'imgsz': 640, 'conf': 0.46, 'nms_iou': 0.7, 'max_det': 300, 'matching_iou': 0.5, 'sha256': '25b412ff18c05e2607f4bc61b872f39826e8d53ffdc7647654c1f419e355d15f', 'val_mAP50': 0.9593002182315967, 'val_mAP50_95': 0.6578987935389327, 'selection': 'China validation only (mAP50-95)', 'local_images_used': False}
